# BLS walkthrough

The Bureau of Labor Statistics publishes the United States' labor and price
statistics -- unemployment, payrolls, CPI, PPI, wages, productivity -- spread
across about seventy survey programs. Every number belongs to a series named by
a composite code such as `LNU04000000`, where each field of the code carries a
piece of the definition.

The surprising part is what the API does *not* have: a search. No endpoint
matches text against series, so nobody guesses an id and nobody looks one up by
description. Discovery runs through the surveys instead, and that is the arc
this notebook follows.

The MCP server must be running; `MCP_URL` selects it (default
`http://localhost:8080/sse`).

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append('../')

from datetime import date

from matplotlib import pyplot

from lib import config
from lib.utils import print_json_vertical
from utils import (
    call_tool,
    list_mcp_tools,
    show_tool_schema,
    show_all_surveys,
    popular_series,
    show_series_catalog,
    show_notices,
    show_observations,
    filter_observations,
    observation_span,
    plot_bls_series,
    suppressed_observations,
)

pyplot.style.use(config.glyfish_style)

## 1. Discovery

Five tools cover BLS, and the split between them says where the difficulty
lies: three are for finding a series (`bls_all_surveys`, `bls_survey_info`,
`bls_popular_series`) and only two return observations.

In [ ]:
await list_mcp_tools(prefix="bls_")

In [ ]:
schema = await show_tool_schema("bls_series_data")

`bls_series_data` is the tool everything else leads to, and its arguments
describe BLS more than they describe the tool.

`series_ids` is a list rather than a string: one call carries up to fifty
series, so batching is the normal way to ask, not an optimization. The range is
given in whole **years** -- BLS has no start date, because an observation is
dated by a period *code* (`M06`, `Q02`, `A01`) rather than a calendar field.
The four booleans all default to off and each changes the shape of the
response: `catalog` attaches the series title and units, `calculations` adds
net and percent changes over trailing spans, `annualaverage` adds aggregate
rows, `aspects` adds secondary measures such as standard errors.

Nothing here accepts a search string, which is the whole problem: the id has to
come from somewhere else.

## 2. Finding a series

That somewhere is the survey list. The path runs survey -> survey metadata ->
popular series -> catalog: pick one of the ~70 programs, ask what it supports,
take the twenty-five series people request most, and fetch those with
`catalog=True` to learn what they are. The popular-series tool returns bare ids
and nothing else, so that last step is what makes them readable.

In [ ]:
surveys = await show_all_surveys(match="Current Population")

In [ ]:
info = await call_tool("bls_survey_info", {"survey_abbreviation": "LN"})
print_json_vertical(info.structuredContent["survey"])

popular_ids = await popular_series("LN")
print(f"\n{len(popular_ids)} popular LN series, as returned:")
print(popular_ids)

In [ ]:
last_year = date.today().year - 1

named = await call_tool("bls_series_data", {
    "series_ids": popular_ids,      # all 25 in one call; the cap is 50
    "start_year": last_year,        # one year wide: we are here for the metadata
    "end_year": last_year,
    "catalog": True,
})

titles = show_series_catalog(named.structuredContent)

## 3. Fetch and plot

`LNU04000000` is the unemployment rate **unadjusted** -- the same measurement
as the headline `LNS14000000` with the seasonal smoothing left off, so the
calendar is still in the numbers.

`bls_series_latest` answers "what is the newest number" in a single call, and
it is the yardstick for the fetch that follows: whatever a full-history request
returns should end there.

It will not. BLS caps a query at twenty years and does not treat a longer
request as an error -- it answers successfully, with a shorter series, and
mentions the fact only in `notices`. Ask for everything back to 1948 and watch
what actually arrives.

In [ ]:
series_id = "LNU04000000"

latest = await call_tool("bls_series_latest", {"series_id": series_id})
print("newest observation BLS has:")
show_observations(latest.structuredContent["series"][0])

everything = await call_tool("bls_series_data", {
    "series_ids": [series_id],
    "start_year": 1948,
    "end_year": date.today().year,
})

print()
show_notices(everything.structuredContent)
print("\nspan returned:", observation_span(everything.structuredContent["series"][0]))

Note *which* twenty years came back: the oldest ones. The response is a
well-formed run of monthly unemployment rates that happens to stop in the
1960s, and nothing but that notice distinguishes it from a complete answer -- a
caller who drops `notices` plots the Eisenhower era and labels it current.

So ask for a window that fits. `annualaverage=True` goes in as well, to show
the other quirk: BLS returns each year's mean as an extra row coded `M13`, and
the server dates `M13` to January 1st, the same date as that year's `M01`. Two
rows land on one date and `period_type` is the only field that separates them,
which is why the plot below filters on it.

What makes the plot worth looking at is the saw-tooth riding on top of the two
shocks: the rate peaks every January, bumps again in June as school leavers
enter the labor force, and bottoms out in the autumn. That yearly ripple is
what the seasonally adjusted headline exists to remove, and it is roughly a
percentage point wide -- larger than most of the month-to-month moves the
headline number gets reported on.

In [ ]:
end_year = date.today().year
start_year = end_year - 19          # 20 years inclusive: exactly the BLS limit

result = await call_tool("bls_series_data", {
    "series_ids": [series_id],
    "start_year": start_year,
    "end_year": end_year,
    "catalog": True,
    "annualaverage": True,
})
payload = result.structuredContent
show_notices(payload)

series = payload["series"][0]
print(f"\n{len(series['observations'])} observations, span {observation_span(series)}")

print(f"\nboth rows dated {last_year}-01-01:")
show_observations(series, on_date=f"{last_year}-01-01")

In [ ]:
monthly = filter_observations(series, period_type="monthly")

# BLS withholds a figure by publishing a non-numeric placeholder rather than
# omitting the row, so the plot can have fewer points than the series has
# observations. Say so, rather than letting the gap pass unremarked.
withheld = suppressed_observations(monthly)
print(f"{len(monthly['observations'])} monthly observations, "
      f"{len(monthly['observations']) - len(withheld)} plottable")
for row in withheld:
    notes = "; ".join(f.get("text") or "" for f in row.get("footnotes") or [])
    print(f"  withheld: {row['date']}  value={row['value']!r}  {notes}")

plot_bls_series(monthly)